# 03 — Deep Learning Experiments (BiLSTM + Attention, Transformer)

> Requires PyTorch (+ Transformers for the BERT model): `pip install torch transformers`.

This notebook trains the BiLSTM-with-attention and the fine-tuned multilingual transformer, compares them against the baseline, and visualizes attention as an explanation.

In [ ]:
import sys; sys.path.insert(0,'../src')
import pandas as pd, numpy as np
from political_sentiment.data.io import load_frame
from political_sentiment.evaluation.metrics import compute_metrics
train = load_frame('../data/processed','train'); val = load_frame('../data/processed','val'); test = load_frame('../data/processed','test')

## BiLSTM + Attention

In [ ]:
from political_sentiment.models.bilstm import BiLSTMAttentionModel
bilstm = BiLSTMAttentionModel(epochs=8, batch_size=64, hidden_dim=128, embedding_dim=128)
bilstm.fit(train, val)
p = bilstm.predict_proba(test['text'].tolist()); preds=[bilstm.labels[i] for i in p.argmax(1)]
compute_metrics(test['label'].tolist(), preds, p, bilstm.labels)

### Attention as explanation
Token attention weights show which words drove the prediction — a faithful, built-in explanation.

In [ ]:
toks, w = bilstm.attention_weights('PartyA plan on the economy is not good at all')
import matplotlib.pyplot as plt
plt.figure(figsize=(8,1.2)); plt.bar(range(len(toks)), w); plt.xticks(range(len(toks)), toks, rotation=45, ha='right'); plt.title('BiLSTM attention'); plt.tight_layout(); plt.show()

## Fine-tuned multilingual transformer (DistilBERT / XLM-R)

In [ ]:
from political_sentiment.models.transformer import TransformerModel
bert = TransformerModel(pretrained_name='distilbert-base-multilingual-cased', epochs=3, batch_size=16)
bert.fit(train, val)
p = bert.predict_proba(test['text'].tolist()); preds=[bert.labels[i] for i in p.argmax(1)]
compute_metrics(test['label'].tolist(), preds, p, bert.labels)

## Model comparison

In [ ]:
# Assumes the three models were trained above / via `make all`.
from political_sentiment.evaluation.plots import plot_model_comparison
# Fill with your measured metrics dicts:
# plot_model_comparison({'baseline':..., 'bilstm':..., 'transformer':...}, '/tmp/cmp.png')

### Explainability with LIME / SHAP
`pip install lime shap`, then:
```python
from political_sentiment.explainability import explain_text
explain_text(bert, 'এই নীতি মোটেও ভালো নয়', method='lime')
```